In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *

In [0]:
customer=spark.read.table("fabric_lh.silver.customers").alias("c")
orders=spark.read.table("fabric_lh.silver.orders").alias("o")
payments=spark.read.table("fabric_lh.silver.payments").alias("p")
support=spark.read.table("fabric_lh.silver.support_trickets").alias("s")
web=spark.read.table("fabric_lh.silver.web_activities").alias("w")

In [0]:
customer360 = (
    customer
    .join(orders, col('c.customer_id')==col('o.Customer_id'), how='left')
    .join(payments, col('c.customer_id')==col('p.Customer_id'), how='left')
    .join(support, col('c.customer_id')==col('s.Customer_id'), how='left')
    .join(web, col('c.customer_id')==col('w.Customer_id'), how='left')
    .select(
        col("c.customer_id"), 
        col("c.name"), 
        col("c.email"), 
        col("c.gender"), 
        col("c.dob"),
        col("c.location"),

        col("o.order_id"), 
        col("o.order_date"), 
        col("p.amount").alias("Order_amount"), 
        col("o.status"),

        col("p.payment_method"), 
        col("p.payment_status"), 
        col("p.amount").alias("Payment_amount"),

        col("s.ticket_id"), 
        col("s.issue_type"), 
        col("s.ticket_date"), 
        col("s.resolution_status"),

        col("w.page_viewed"), 
        col("w.device_type"), 
        col("w.session_time")
    )
)

In [0]:
display(customer360)

customer_id,name,email,gender,dob,location,order_id,order_date,Order_amount,status,payment_method,payment_status,Payment_amount,ticket_id,issue_type,ticket_date,resolution_status,page_viewed,device_type,session_time
1003,null,tom@outlook.com,Male,null,San Francisco,2003,2022-07-18,250.0,Shipped,Paypal,Success,250.0,T003,Payment Issue,2022-07-20,Closed,/products/bags,Windows,2022-07-19
1015,Karan Singh,karan@yahoo.co.in,Male,1993-07-07,Delhi,2015,2022-07-26,0.0,Pending,Netbanking,Success,0.0,T015,Payment Error,2022-07-26,Pending,/products/electronics,Windows,2022-07-26
1002,Sara Khan,sara.khan@gmail.com,Female,1988-06-12,Mumbai,2002,2022-07-15,null,Pending,null,null,null,T002,Delivery Delay,2022-07-19,Pending,/products/shoes,Ios,2022-07-18
1007,Meena Kumari,meena@rediffmail.com,Female,1995-04-10,Chennai,2007,2022-07-21,300.0,Shipped,Credit Card,Success,300.0,T007,Delivery Delay,2022-07-21,Closed,/products/electronics,Windows,2022-07-21
1008,Rajesh,rajesh@yahoo.com,Male,1990-01-01,Hyderabad,null,null,0.0,null,Paypal,Failed,0.0,T008,Product Damaged,2022-07-22,Resolved,/products/shoes,Macos,2022-07-22
1011,Ravi Kumar,ravi.k@inbox.com,Male,1988-05-25,Pune,null,null,null,null,null,null,null,null,null,null,null,/products/shoes,Ios,2022-07-23
1001,John Doe,john.doe@gmail.com,Male,1990-05-12,New York,2001,2022-07-15,120.5,Complete,Credit Card,Success,120.5,T001,Payment Issue,2022-07-19,Closed,/home,Android,2022-07-18
1006,Amit Patel,amit123@gmail.com,Male,1992-08-19,Surat,2006,2022-07-21,500.0,Complete,Wallet,Success,500.0,T006,Refund Request,2022-07-21,Pending,/home,Ios,2022-07-20
1009,Sophie,sophie@gmail,Female,1991-02-02,Mumbai,2009,2022-07-22,null,Cancelled,null,null,null,T009,Login Issue,2022-07-23,Pending,null,null,null
1005,Priya R,priya.r@hotmail.com,Female,1993-11-23,Delhi,2005,2022-07-20,null,Failed,null,null,null,null,null,null,null,/checkout,Android,2022-07-20


In [0]:
customer360.write.format("delta").mode("overwrite").saveAsTable("fabric_lh.gold.customer360")